# SPDX SBOM Analysis Jupyter Notebook

This jupyter notebook was developed as part of the supporting materials for the 2024 CISA SBOM Plugfest. The purpose of this notebook is to ingest SPDX SBOMs, and extract information from them as a batch, and export summary and component information in csv files. 

**Packages:**
- *os*: for traversing file directories
- *json*: deserializing json files
- *pandas*: building dataframes
- *networkx*: graph algorithms (depth, paths)
- *matplotlib.pyplot*: optional plotting
- *re*: regular expression parsing
- *networkx.drawing.nx_pydot.graphviz_layout*: optional plotting
- *pathlib*: creating directories and path objects
- *collections.defaultdict*: instantitating dictionaries

*shared_functions* is a separate file hosting a number of functions needed for processing SBOMs. Our intention was to migrate many functions in notebooks into separate python files and to convert notebooks into python scripts, but we ran out of time. Note: shared_functions.py should be co-located with this notebook.

## Recommended Directory Structure   
This directory structure was used to facilitate processing of .json sboms of each type (CycloneDX, SPDX). There are other more efficient ways to do this, but this is the methodology we used. As written, these notebooks should be run from the ```code``` location. Additionally, although only two targets are illustrated here, this structure should be replicated for sboms corresponding to additional targets.
```
📂project
┣ 📂code
┃ ┣ 📜cyclonedx_analyzer.ipynb
┃ ┣ 📜shared_functions.py
┃ ┗ 📜spdx_analyzer.ipynb
┣ 📂outputs
┣ 📂submissions_by_target
┃ ┣ 📂target_1
┃ ┃ ┣ 📂build
┃ ┃ ┃ ┣ 📂cyclone
┃ ┃ ┃ ┃ ┗ 📜cdx_build_target_2_sbom_1.json
┃ ┃ ┃ ┗ 📂spdx
┃ ┃ ┃   ┗ 📜spdx_build_target_2_sbom_1.json
┃ ┃ ┗ 📂source
┃ ┃   ┣ 📂cyclone
┃ ┃   ┃ ┗ 📜cdx_source_target_2_sbom_1.json
┃ ┃   ┗ 📂spdx
┃ ┃     ┗ 📜spdx_source_target_2_sbom_1.json
┃ ┗ 📂target_2
┃   ┣ 📂build
┃   ┃ ┣ 📂cyclone
┃   ┃ ┃ ┗ 📜cdx_build_target_2_sbom_1.json
┃   ┃ ┗ 📂spdx
┃   ┃   ┗ 📜spdx_build_target_2_sbom_1.json
┃   ┗ 📂source
┃     ┣ 📂cyclone
┃     ┃ ┗ 📜cdx_source_target_2_sbom_1.json
┃     ┗ 📂spdx
┃       ┗ 📜spdx_source_target_2_sbom_1.json
┗ 📂baseline_sboms
     ┣ 📂target_1
     ┃ ┗ 📜target_1_baseline_sbom_1.json
     ┗ 📂target_2
       ┗ 📜target_2_baseline_sbom_1.json
```

## Outputs

After we produced data for the SBOMs in CycloneDX format, for the purposes of the 2024 Plugfest, we decided to only produce merged files from which we did the remainder of the analysis on the (1) SBOM metadata and (2) high-level component data. Data on individual components within SPDX SBOMs was not extracted in support of the 2024 Plugfest, but can be through repurposing other code or writing code from scratch.

After running all cells, you will produce:
- ```merged_spdx_component_files.csv```: file that combines all data which summarizes how many SPDX SBOMs of type (target; source or build) each (component, version) was found in. There will be data for SBOMs of each phase for each target, but this was simply merged together and exported.

- ```merged_spdx_summary_files.csv```: a file that combines summarizing top-level information about each sbom (primarily, minimum elements). There will be data for SBOMs of each phase for each target, but this was simply merged together and exported.

## Import Statement

In [ ]:
import os
import json
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import re
from networkx.drawing.nx_pydot import graphviz_layout
from pathlib import Path
from collections import defaultdict, deque

from shared_functions import *

## Custom Functions for Parsing SPDX SBOMs

In [ ]:
def calculate_depth(packages, relationships, plotting = False):
    print("Calculating depth, this might take a while...")
    sbom_target = ""
    graph = nx.DiGraph()
    #for package in packages:
    #    graph.add_node(package['SPDXID'])

    node_mapping = {}
    relationship_types = []
    count = 0
    for relationship in relationships:
        is_depends = False
        is_dependency = False
        relationship_types.append(relationship['relationshipType'])
        if relationship['spdxElementId'] == 'SPDXRef-DOCUMENT':
            #source = relationship['spdxElementId']
            sbom_target = relationship['relatedSpdxElement']
            continue
        else:
            is_depends = bool(re.search(r"(?i)depends", relationship['relationshipType']))
            is_dependency = bool(re.search(r"(?i)dependency", relationship['relationshipType']))
            if relationship['relationshipType'] == 'OTHER':
                continue
            elif is_depends:
                source = relationship['spdxElementId']
                target = relationship['relatedSpdxElement']
                if source not in node_mapping.keys() and target not in node_mapping.keys():
                    source_label = f"S_{count}"
                    node_mapping[source] = source_label
                    count = count + 1
                    target_label = f"S_{count}"
                    node_mapping[target] = target_label
                    count = count + 1
                elif source not in node_mapping.keys() and target in node_mapping.keys():
                    source_label = f"S_{count}"
                    node_mapping[source] = source_label
                    count = count + 1
                    target_label = node_mapping[target]
                elif source in node_mapping.keys() and target not in node_mapping.keys():
                    target_label = f"S_{count}"
                    node_mapping[target] = target_label
                    count = count + 1
                    source_label = node_mapping[source]
                else:
                    source_label = node_mapping[source]
                    target_label = node_mapping[target]
                graph.add_edge(source_label, target_label)
            elif is_dependency:
                target = relationship['spdxElementId']
                source = relationship['relatedSpdxElement']
                if source not in node_mapping.keys() and target not in node_mapping.keys():
                    source_label = f"S_{count}"
                    node_mapping[source] = source_label
                    count = count + 1
                    target_label = f"S_{count}"
                    node_mapping[target] = target_label
                    count = count + 1
                elif source not in node_mapping.keys() and target in node_mapping.keys():
                    source_label = f"S_{count}"
                    node_mapping[source] = source_label
                    count = count + 1
                    target_label = node_mapping[target]
                elif source in node_mapping.keys() and target not in node_mapping.keys():
                    target_label = f"S_{count}"
                    node_mapping[target] = target_label
                    count = count + 1
                    source_label = node_mapping[source]
                else:
                    source_label = node_mapping[source]
                    target_label = node_mapping[target]
                graph.add_edge(source_label, target_label)

    root_nodes_and_degrees = [[node, graph.out_degree(node)] for node  in graph.nodes if graph.in_degree(node) == 0]
    root_nodes = [pair[0] for pair in root_nodes_and_degrees]

    leaf_nodes = [node for node in graph.nodes if graph.out_degree(node) == 0]

    max_depth = 0
    for root in root_nodes:
        for leaf in leaf_nodes:
            try:
                path_length = nx.shortest_path_length(graph, source = root, target = leaf)
                max_depth = max(max_depth, path_length)
            except Exception as e:
                # maybe there are no paths?
                print(e)
                continue
            
    # This block of code is for plotting graphs, if plotting = True. Default is False.
    if plotting:
        pos = graphviz_layout(graph, prog="dot")
        plt.figure(figsize = (40,40))
        nx.draw(graph, pos, with_labels = True, font_size = 35)
        plt.show()
    
    return max_depth, sbom_target

In [ ]:
def component_analysis(sbom):
    package_info = []
    packages = sbom.get("packages", [])
    purl = "None"
    package_info_dedupe = []
    for package in packages:
        try:
            exRef = package.get("externalRefs")
            for ref in exRef:
                if ref['referenceType'] == "purl":
                    purl = ref['referenceLocator']
                else:
                    continue
        except Exception as e:
            print(e)
            continue
        package_info.append({
            "component_name": package.get("name"),
            "version": package.get("versionInfo"),
            "purl": purl
        })
        package_info_dedupe = [dict(t) for t in {tuple(d.items()) for d in package_info}]
    return package_info_dedupe

In [ ]:
def parse_spdx_data(sbom, plotting = False):
    """ 
    Extract the meaningful information
    """
    creation_info_sbom = sbom.get("creationInfo", {'licenseListVersion': 'False', 'creators': "False", 'created': 'False'})
    creation_info_parsed = {"person": "False", "tool": "False", "organization": "False"}
    sbom_supplier = 'None'
    if type(creation_info_sbom) is dict:
        for string in creation_info_sbom['creators']:
            if "person" in string.lower():
                creation_info_parsed["person"] = "True"
                sbom_supplier = "True"
            if "organization" in string.lower():
                creation_info_parsed["organization"] = "True"
                sbom_supplier = "True"
            if "tool" in string.lower():
                creation_info_parsed["tool"] = "True"
    hash_info = sbom.get("externalDocumentRefs", {"algorithm": "None", "checksumValue": "None"}) #some sboms do this?
    if type(hash_info) is list:
        hash_info = hash_info[0]
    details = {
            "name": sbom.get("name", None),
            "spdx_version": sbom.get("spdxVersion", None),
            "data_license": sbom.get("dataLicense", None),
            "document_namespace": sbom.get("documentNamespace", None),
            "creation_date": creation_info_sbom.get("created", None),
            "creator_person": creation_info_parsed.get("person", None),
            "creator_org": creation_info_parsed.get("organization", None),
            "creator_tool": creation_info_parsed.get("tool", None),
            "sbom_supplier": sbom_supplier,
            "packages": [],
            "depth": 0,
            "hash_alg": hash_info.get("algorithm", None),
            "hash": hash_info.get("checksumValue", None)
    }

    # get package info
    packages = sbom.get("packages", [])
    number_of_components = len(packages)
    details['number_of_components'] = number_of_components

    relationships = sbom.get("relationships", [])
    depth, target = calculate_depth(packages, relationships)

    details["depth"] = depth
    details['target_spdx_id'] = target

    #Can we use the target ID to pull out the info for the target? Let's try...
    packages = sbom.get("packages", [])
    for package in packages:
        if package.get("SPDXID") == target:
                details["target_name"]= package.get("name")
                details["target_version"]= package.get("versionInfo")
                details["target_license"]= package.get("licenseConcluded")
                details["supplier"] = package.get("supplier")
                details["originator"] = package.get("originator")
                details["primaryPackagePurpose"] = package.get("primaryPackagePurpose")
                try:
                    hash_info = package.get("checksums")[0]
                    details['hash_alg'] = hash_info['algorithm']
                    details['hash'] = hash_info['checksumValue']
                except Exception as e:
                    print(e)
                    continue

    package_info = component_analysis(sbom)
    details['components'] = package_info

    return details

In [ ]:
def parse_spdx_file(filepath, plotting = False):
    """
    Import the sbom
    """
    try:
        if filepath.endswith('.json'):
            with open(filepath, 'r') as file:
                print(f"Processing {filepath}")
                sbom = json.load(file)
                return parse_spdx_data(sbom, plotting = False)
        else:
            print(f"Whoops! I don't support this filetype yet: {filepath}")
    except Exception as e:
        print(f"Failed to parse {filepath} : {e}")
        return None

In [ ]:
def process_directories(base_dir):
    """
    Process files in the directory structure
    """
    all_summary = []
    all_components = []
    file_counts = {}
    for root, _, files in os.walk(base_dir):
        file_number = 0
        for file in files:
            filepath = os.path.join(root, file)
            if filepath.endswith(('.json')) and 'spdx' in filepath.lower():
                path_parts = Path(filepath).parts
                file = path_parts[-1]
                build_or_source = path_parts[-3]
                target_truth = path_parts[-4]
                file_number = file_number + 1
                file_counts[(target_truth, build_or_source)] = file_number
                target = "unknown"
                phase = "unknown"
                try:
                    target = path_parts[path_parts.index("submissions_by_target") + 1]
                    phase = "build" if "build" in path_parts else "source"
                except ValueError:
                    print("Failed to get target and phase")

                summary = parse_spdx_file(filepath, plotting = False)
                if summary:
                    all_summary.append({
                        "File": file,
                        "Total Components": summary.get("number_of_components"),
                        "Max Depth": summary.get("depth"),
                        "Max Breadth": None,
                        "Singletons": None,
                        "Target Type": summary.get("primaryPackagePurpose"),
                        "Target Name": summary.get("name"),
                        "Target Hash": summary.get("hash"),
                        "Target Version": summary.get("target_version"),
                        "Target Supplier": summary.get("supplier"),
                        "Target in Dep Tree": None,
                        "SBOM Format": "SPDX",
                        "SBOM Format Version": summary.get("spdx_version"),
                        "SBOM Serial Number": summary.get("document_namespace"),
                        "SBOM Version": None,
                        "SBOM Supplier": summary.get("sbom_supplier"),
                        "SBOM Timestamp": summary.get("creation_date"),
                        "Target Licenses": summary.get("target_license"),
                        "SBOM Author": summary.get("creator_person"),
                        "SBOM Phase": None,
                        "Target": target_truth,
                        "Build or Source": build_or_source
                    })
                    for i in range(len(summary['components'])):
                        summary['components'][i]['target'] = target_truth,
                        summary['components'][i]['build_or_source'] = build_or_source,
                        summary['components'][i]['filename'] = file
                    all_components.append(summary['components'])


    return all_summary, all_components, file_counts

In [ ]:
# process the SBOMs
base_dir = "../submissions_by_target"
records, components, file_counts = process_directories(base_dir)

In [ ]:
# Build the dataframes of data

df_components = pd.DataFrame()
for c in components:
    df_components_part= pd.DataFrame(c)
    df_components = pd.concat([df_components, df_components_part])
df_components['target'] = df_components['target'].astype(str)
df_components['target'] = df_components['target'].replace(r"[(),']", '', regex = True)
df_components['build_or_source'] = df_components['build_or_source'].astype(str)
df_components['build_or_source'] = df_components['build_or_source'].replace(r"[(),']", '', regex = True)

df_components = df_components.convert_dtypes()
df_components.loc[:, 'number_of_files']= df_components.set_index(['target', 'build_or_source']).index.map(file_counts)

#Path("../outputs/spdx_sbom_data/component_similarity").mkdir(parents=True, exist_ok=True)
#Path("../outputs/spdx_sbom_data/summary_files").mkdir(parents=True, exist_ok=True)

#component_output_file = "../outputs/spdx_sbom_data/summary_files/merged_spdx_component_files.csv"
#df_components.to_csv(component_output_file, index = False)
#df_components.to_csv(component_output_file)


In [ ]:
Path("../outputs/spdx_sbom_data/component_similarity").mkdir(parents=True, exist_ok=True)

component_output_file = "../outputs/spdx_sbom_data/component_similarity/merged_spdx_component_files.csv"
df_components.to_csv(component_output_file, index = False)
df_components.to_csv(component_output_file)

In [ ]:
summary_df = pd.DataFrame(records)

Path("../outputs/spdx_sbom_data/summary_files").mkdir(parents=True, exist_ok=True)

summary_output_file = "../outputs/spdx_sbom_data/summary_files/merged_spdx_summary_files.csv"
summary_df.to_csv(summary_output_file, index = False)

## Baseline SPDX Analysis

In [ ]:
def process_baseline_directories(base_dir):
    """
    Process files in the directory structure
    """
    all_summary = []
    all_components = []
    file_counts = {}
    for root, _, files in os.walk(base_dir):
        file_number = 0
        for file in files:
            filepath = os.path.join(root, file)
            if filepath.endswith(('.json')) and 'spdx' in filepath.lower():
                sbomFile = sbomInfoFromFile(filepath)
                file = os.path.basename(filepath)
                build_or_source = sbomFile.phase.value
                target_truth = sbomFile.target.value
                file_number = file_number + 1
                file_counts[(target_truth, build_or_source)] = file_number
                target = sbomFile.target.value
                phase = sbomFile.phase.value

                summary = parse_spdx_file(filepath)
                if summary:
                    all_summary.append({
                        "File": file,
                        "Total Components": summary.get("number_of_components"),
                        "Max Depth": summary.get("depth"),
                        "Max Breadth": None,
                        "Singletons": None,
                        "Target Type": summary.get("primaryPackagePurpose"),
                        "Target Name": summary.get("name"),
                        "Target Hash": summary.get("hash"),
                        #"hash_alg": summary.get("hash_alg"),
                        "Target Version": summary.get("target_version"),
                        "Target Supplier": summary.get("supplier"),
                        "Target in Dep Tree": None,
                        "SBOM Format": "SPDX",
                        "SBOM Format Version": summary.get("spdx_version"),
                        "SBOM Serial Number": summary.get("document_namespace"),
                        "SBOM Version": None,
                        "SBOM Supplier": summary.get("sbom_supplier"),
                        "SBOM Timestamp": summary.get("creation_date"),
                        "Target Licenses": summary.get("target_license"),
                        "SBOM Author": summary.get("creator_person"),
                        "SBOM Phase": None,
                        #"name": summary.get("name"),
                        "Target": target,
                        "Build or Source": build_or_source
                        #"data_license": summary.get("data_license"),
                        #"creator_org": summary.get("creator_org"),
                        #"target_spdx_id": summary.get("target_spdx_id"),
                        #"originator": summary.get("originator"),
                    })
                    for i in range(len(summary['components'])):
                        summary['components'][i]['target'] = target_truth
                        summary['components'][i]['build_or_source'] = build_or_source
                    all_components.append(summary['components'])


    return all_summary, all_components, file_counts

In [ ]:
baseline_records, baseline_components, baseline_file_counts = process_baseline_directories(sbom_baseline_dir)

In [ ]:
baseline_summary_df = pd.DataFrame(baseline_records)
baseline_df_components = pd.DataFrame()
for c in baseline_components:
    df_components_part= pd.DataFrame(c)
    baseline_df_components = pd.concat([baseline_df_components, df_components_part])
baseline_df_components['target'] = baseline_df_components['target'].astype(str)
baseline_df_components['target'] = baseline_df_components['target'].replace(r"[(),']", '', regex = True)
baseline_df_components['build_or_source'] = baseline_df_components['build_or_source'].astype(str)
baseline_df_components['build_or_source'] = baseline_df_components['build_or_source'].replace(r"[(),']", '', regex = True)

baseline_df_components = baseline_df_components.convert_dtypes()

In [ ]:
baseline_df_components.loc[:, 'number_of_files']= baseline_df_components.set_index(['target', 'build_or_source']).index.map(baseline_file_counts)

In [ ]:
baseline_component_output_file = "baseline_spdx_component_analysis_v2.csv"
baseline_df_components.to_csv(baseline_component_output_file, index = False)
baseline_df_components.to_csv(baseline_component_output_file)

In [ ]:
baseline_summary_output_file = "baseline_spdx_summary.csv"
baseline_summary_df.to_csv(baseline_summary_output_file, index = False)